# Climate, Statistics, and Probability
### An interactive introduction to climate variability and climate change

**Audience:** Introductory Weather & Climate  
**Time:** ~60–90 minutes (choose sections for a 50-minute class)  
**Data:** Synthetic examples, deliberately constructed for teaching; **not observed climate records**.

**Learning goals:** Interpret climatological normals and anomalies; compare means and variability; identify trends; understand shifting distributions and extreme-event probabilities; explain return periods.

**Instructor workflow:** Run cells in order. Before each plot, ask students to predict what will happen. Change the parameters in the `SETTINGS` cells and rerun to test their predictions. No internet connection or special datasets required.

## Setup
Run this cell first. Requires `numpy`, `pandas`, and `matplotlib` (common in Anaconda/Jupyter environments). Every random experiment uses a fixed seed so figures are reproducible.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (10, 4.8), "font.size": 11})
rng = np.random.default_rng(2026)
print("Ready! All datasets in this notebook are synthetic teaching examples.")

## 1 — Weather versus climate: What does “normal” mean?

**Ask first:** If the average July high is 90°F, how often must the daily high equal 90°F?

The next example generates 30 Julys of daily high temperatures with a stable underlying climate. The 30-year average is a *reference value*, not a temperature that every day should reach.

In [ ]:
# SETTINGS — try changing the average and day-to-day variability
july_mean_f = 90
july_sd_f = 6

july_highs = rng.normal(july_mean_f, july_sd_f, size=(30, 31))
normal = july_highs.mean()
fig, ax = plt.subplots()
ax.hist(july_highs.ravel(), bins=28, edgecolor="white", alpha=0.85)
ax.axvline(normal, color="black", linewidth=2, label=f"30-year July mean: {normal:.1f}°F")
ax.set(xlabel="Daily July high (°F)", ylabel="Number of days", title="Daily weather varies around a climate normal")
ax.legend(); plt.show()
print(f"Mean: {normal:.1f}°F | Median: {np.median(july_highs):.1f}°F | Standard deviation: {july_highs.std():.1f}°F")

**Discuss:** What does the width of the histogram tell us? Would a 100°F day necessarily be evidence of climate change? Why or why not?

> **Note:** This simplified simulation treats daily observations as independent and uses a normal distribution. Real daily temperatures are temporally correlated and may not be normally distributed.

## 2 — Same average, different variability

**Prediction:** Can two places have the same average high temperature but different frequencies of very hot days?

Change the standard deviations while keeping the means equal.

In [ ]:
# SETTINGS
shared_mean = 85
sd_city_a = 3
sd_city_b = 10
hot_threshold = 95

r = np.random.default_rng(17)
a = r.normal(shared_mean, sd_city_a, 20000)
b = r.normal(shared_mean, sd_city_b, 20000)
fig, ax = plt.subplots()
bins = np.arange(45, 126, 2)
ax.hist(a, bins=bins, density=True, alpha=0.5, label="City A: low variability")
ax.hist(b, bins=bins, density=True, alpha=0.5, label="City B: high variability")
ax.axvline(hot_threshold, color="black", linestyle="--", label=f"{hot_threshold}°F threshold")
ax.set(xlabel="Daily high (°F)", ylabel="Relative frequency (density)", title="Equal means do not imply equal extremes")
ax.legend(); plt.show()
print(f"City A above {hot_threshold}°F: {100*np.mean(a > hot_threshold):.1f}%")
print(f"City B above {hot_threshold}°F: {100*np.mean(b > hot_threshold):.1f}%")

**Discuss:** Which city has more extreme heat? Which city has more unusually cool days? Does an average alone describe a climate adequately?

## 3 — Climate anomalies and reference periods

An **anomaly** is an observed value minus the average over a specified reference period. Here we simulate annual temperatures from 1981–2025 with a warming trend and year-to-year noise.

**Prediction:** Does changing the reference period change the *underlying temperatures* or only the anomalies?

In [ ]:
# SETTINGS
warming_per_decade_f = 0.45
baseline_start, baseline_end = 1991, 2020  # Try 1981, 2010

years = np.arange(1981, 2026)
r = np.random.default_rng(41)
annual_t = 62 + warming_per_decade_f * (years - 1981)/10 + r.normal(0, 0.7, len(years))
mask = (years >= baseline_start) & (years <= baseline_end)
assert mask.sum() >= 10, "Choose a baseline with at least 10 years in the record."
baseline = annual_t[mask].mean()
anomaly = annual_t - baseline
fig, ax = plt.subplots()
ax.bar(years, anomaly, color=np.where(anomaly >= 0, "tomato", "steelblue"))
ax.axhline(0, color="black", linewidth=1)
ax.set(xlabel="Year", ylabel="Annual temperature anomaly (°F)",
       title=f"Synthetic annual anomalies relative to {baseline_start}–{baseline_end}")
plt.show()
print(f"Baseline average: {baseline:.2f}°F | Latest anomaly: {anomaly[-1]:+.2f}°F")

**Discuss:** Why can a year have a negative anomaly even when the long-term climate is warming? Why must a chart of anomalies identify its baseline?

## 4 — Trends versus year-to-year variability

A trend summarizes systematic change over time, while individual years can be warmer or cooler than the trend. A fitted line is a *description of the data*, not by itself proof of what caused the change.

**Prediction:** How might a trend estimated from only five years differ from one estimated over the full record?

In [ ]:
# SETTINGS
short_window_years = 5  # Try 10, 15, or 30

slope, intercept = np.polyfit(years, annual_t, 1)
recent = years >= years.max() - short_window_years + 1
short_slope, short_intercept = np.polyfit(years[recent], annual_t[recent], 1)
fig, ax = plt.subplots()
ax.plot(years, annual_t, "o-", alpha=0.55, markersize=4, label="Annual temperatures")
ax.plot(years, slope*years + intercept, linewidth=2.5, label="Full-record linear trend")
ax.plot(years[recent], short_slope*years[recent] + short_intercept,
        linewidth=3, label=f"Last {short_window_years} years trend")
ax.set(xlabel="Year", ylabel="Annual mean temperature (°F)", title="Short-term fluctuations and long-term change")
ax.legend(); plt.show()
print(f"Full-record slope: {slope*10:+.2f} °F/decade")
print(f"Last {short_window_years} years slope: {short_slope*10:+.2f} °F/decade")

**Discuss:** Is a short downward trend inconsistent with a longer upward trend? What is the danger of selecting only a convenient start and end year?

> **Statistical caution:** Real climate time series often have autocorrelation. Formal confidence intervals and significance tests need methods that account for it; ordinary linear fits alone do not establish attribution.

## 5 — Shifting distributions and extreme heat

This experiment holds variability fixed while increasing the average. Observe how the probability of crossing a fixed heat threshold changes.

**Prediction:** Does shifting the mean by 3°F necessarily increase the extreme-heat probability by only a small amount?

In [ ]:
# SETTINGS
historical_mean = 85
warming_f = 3  # Try 0, 1, 3, 5
std_f = 6
extreme_threshold = 95

r = np.random.default_rng(123)
noise = r.normal(0, std_f, 100000)
historical = historical_mean + noise
warmer = historical_mean + warming_f + noise
fig, ax = plt.subplots()
bins = np.arange(60, 115, 1)
ax.hist(historical, bins=bins, density=True, histtype="step", linewidth=2, label="Historical")
ax.hist(warmer, bins=bins, density=True, histtype="step", linewidth=2, label="Warmer climate")
ax.axvline(extreme_threshold, color="black", linestyle="--", label=f"Extreme threshold ({extreme_threshold}°F)")
ax.set(xlabel="Daily high (°F)", ylabel="Probability density", title="A change in the mean changes tail probabilities")
ax.legend(); plt.show()
p_old = np.mean(historical >= extreme_threshold)
p_new = np.mean(warmer >= extreme_threshold)
print(f"Historical: {p_old:.1%} of days exceed the threshold")
print(f"Warmer:     {p_new:.1%} of days exceed the threshold")
print(f"Risk ratio: {p_new/p_old:.2f}×" if p_old > 0 else "Historical sample had no exceedances")

**Discuss:** Why can cold days still happen in a warmer climate? What would change if the spread also increased? Are real temperature distributions always symmetrical?

> This is a controlled illustration of a mean shift, **not** a forecast of actual heat-wave frequency. Real extremes depend on season, location, persistence, and other changes in the distribution.

## 6 — Return periods: A “100-year event”

An event with a **1% annual exceedance probability** is often called a “100-year event.” That does **not** mean it occurs on a fixed 100-year schedule.

If each year's event occurrence is independent and the annual probability stays constant at *p*, the probability of at least one event over *n* years is:

\[P(\text{at least one}) = 1-(1-p)^n\]

**Prediction:** What is the probability of at least one 100-year event during a 30-year mortgage?

In [ ]:
# SETTINGS
annual_probability = 0.01  # Try 0.02 or 0.05
exposure_years = 30         # Try 1, 10, 50, or 100

prob_at_least_one = 1 - (1 - annual_probability)**exposure_years
print(f"Annual probability: {annual_probability:.1%}")
print(f"Return period (if probability is stationary): {1/annual_probability:.0f} years")
print(f"Chance of at least one in {exposure_years} years: {prob_at_least_one:.1%}")
ns = np.arange(1, 101)
fig, ax = plt.subplots()
ax.plot(ns, 100*(1-(1-annual_probability)**ns), linewidth=2)
ax.scatter([exposure_years], [100*prob_at_least_one], s=70, zorder=3)
ax.set(xlabel="Number of years", ylabel="Chance of ≥1 event (%)",
       title="Cumulative probability of at least one event", xlim=(1, 100), ylim=(0, 100))
ax.grid(alpha=0.25); plt.show()

**Discuss:** Can two 100-year floods happen in consecutive years? What happens to the return period if the annual probability changes as climate changes?

> **Assumptions matter:** This formula assumes independent annual occurrences and an unchanging annual probability. These assumptions may fail for real hazards, especially under nonstationary climate conditions.

## 7 — Putting it together: Interpret a climate claim

**Think–pair–share (5–8 minutes)**

A news article says: “This region's average summer temperature increased by 2°F, so extreme heat should be only 2°F worse.”

1. What does the statement get right?
2. What does it miss about probability distributions?
3. What other information would you want (variability, thresholds, baseline, location, time period)?
4. Does the occurrence of a single extreme event establish its cause?

**Exit ticket:** In two or three sentences, explain why an increase in average temperature can increase the frequency of extreme heat by much more than you might expect.

---
## Instructor notes / suggested answers

- **1:** A mean is a summary of many observations; individual days vary around it.
- **2:** Higher variance produces more observations in both tails when means are held equal and distributions are otherwise comparable.
- **3:** Changing the baseline shifts anomaly values but does not alter observed temperatures or their physical trend.
- **4:** Short windows can give misleading trend estimates because of internal variability and sampling; a linear fit does not establish causation.
- **5:** The probability above a fixed threshold can increase sharply as a distribution shifts, even when its spread stays constant.
- **6:** A 1% annual event has a ~26% chance of occurring at least once in 30 years under the stated assumptions. Consecutive events are possible.
- **7:** Changes in mean, spread, and tail shape all matter. Climate attribution requires more than observing a single extreme.

**Optional extension:** Replace the synthetic annual series with a quality-controlled observed station or regional dataset. Discuss missing values, station moves, urbanization, the baseline, seasonality, and whether the data are representative before estimating trends.

**Suggested pacing:** Sections 1–3 (20 min), sections 4–5 (20 min), section 6 (10 min), discussion/exit ticket (5–10 min).